# p4a 执行轨迹观测

前缀层面的问题已在 `01_session_classes.ipynb` 完成实验。

本笔记选择 961 份 sessions ，观察和检测 agent 实际走出来的步骤有多少是共通的、从哪里开始分叉。

观测集是那 961 份。选它的理由见 `docs/experiments/e01-p4a-trajectory.md` §1.3：组内前缀已构造性同质，观察到的分叉可以归因于轨迹本身。

## 1. 筛出观测集

判据是三元组 `(工具Δ, 目录树, delivery)`，取 `(-3882, fb389653, pointer)`，再叠加 s0 的纳入过滤。

In [ ]:
import nbio
import pandas as pd

pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)
nbio.banner()

In [ ]:
# 观测集判据只在 e01/n0_fix_timestamp.py 里定义一次，notebook 不抄。
from e01.n0_fix_timestamp import GROUP

w = nbio.wide()                                   # s0 × s0b × s2 × s3 × s4
sel = w.included.copy()
for k, v in GROUP.items():
    sel &= w[k] == v
g = w[sel].copy()
g["day"] = pd.to_datetime(g.created_at).dt.tz_convert("Asia/Shanghai").dt.date

assert len(g) == 961, f"观测集份数不是 961 而是 {len(g)}，判据或产物变了"
assert g.sysprompt_chars.nunique() == 1, "组内 systemPrompt 长度不唯一"
assert (g.family == "extract").all(), "组内混入了非 extract"

print(GROUP)
print(f"观测集 {len(g)} 份 | {g.paper_id.nunique()} 篇论文 | {g.day.min()} .. {g.day.max()}")
print(f"sysprompt_chars {g.sysprompt_chars.iloc[0]} | 累计 prefill {g.sum_input.sum():,.0f} tok "
      f"（占纳入集 {g.sum_input.sum() / w[w.included].sum_input.sum():.1%}）")

筛完先自查同质性：组内首步 prompt 的跨度就是这一组"前缀有多齐"的直接度量。

In [ ]:
fs = g.first_step_input
print(f"首步 prompt token: min {fs.min()} max {fs.max()}，跨度 {fs.max() - fs.min()} tok")
print(f"时间戳之后被切断的字符 p50: {g.poisoned_tail_chars.median():.0f}")
print()
print("组内仍在变的列（互异 > 1）：")
vary = {c: g[c].nunique() for c in
        ["axis_harness", "axis_tree", "axis_agents", "axis_skills", "axis_timestamp",
         "sysprompt_chars", "tools_tok_rel_ref", "delivery", "pointer_layout", "pid_year"]
        if g[c].nunique() > 1}
print(vary)

只有时间戳在变，其余各轴组内为常数。这正是 §1.3 说的"L2 按构造成立"。

In [ ]:
desc = (g[["n_steps", "n_tools", "peak_input", "sum_input", "amplification",
           "n_external", "n_bash", "n_read", "n_edit", "n_todo", "n_injections"]]
        .describe(percentiles=[.1, .25, .5, .75, .9]).T
        .drop(columns=["count"]).round(1))
desc

## 2. 归一化：固定时间戳

组内只有时间戳一轴在变。把它换成一个常量，systemPrompt 在 961 份之间就逐字相同。

这一步由 `make n0` 产出，判据、常量与自检都在 `e01/n0_fix_timestamp.py`。产物
`data/processed/e01/n0_fixed_time/` 与原始语料同构，逐字节相同，只差 systemPrompt
里那 24 个字符；上下文流只来自 `wire.jsonl`，故只复制它和 `state.json`，53 份带子
agent 的，子 agent wire 同样处理。

流里其余像时间戳的东西是内容不是时钟 —— arXiv 的 `published`、GitHub 的 commit
date、论文的 `submitted` 字段 —— 不动。`state.json` 的 `createdAt` 也不动，它不进
上下文，改了会毁掉时序。

In [ ]:
n0 = nbio.summary("n0")
ck = n0["checks"]
print(f"{n0['corpus']}  {n0['n_sessions']} 份 / {n0['n_papers']} 篇  {n0['corpus_bytes'] / 1e6:.0f} MB")
print(f"时间戳 {n0['orig_timestamp_span']['min']} .. {n0['orig_timestamp_span']['max']}"
      f" -> {n0['fixed_timestamp']}")
print(f"systemPrompt {ck['n_distinct_sysprompt_before']} 个互异值 -> "
      f"{ck['n_distinct_sysprompt_after']} 个（md5 {ck['sysprompt_md5_after'][:8]}）")
print(f"带子 agent 的 session {n0['n_with_subagents']} 份，子 agent wire {n0['n_subagent_wires']} 个")

# 脚本自己选的观测集必须与本笔记筛出的是同一批，否则两边口径已经漂了
ix = nbio.load("n0")
assert set(ix.sid) == set(g.sid), "n0 产物的 sid 集合与本笔记的观测集不一致"
print("✓ 产物与本笔记的观测集是同一批 session")